In [1]:
!pip install opencv-python psutil pynvml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.9 MB/s eta 0:00:00


In [11]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import psutil
import os

# GPU monitoring
try:
    import pynvml
    pynvml.nvmlInit()
    GPU_AVAILABLE = True
except:
    GPU_AVAILABLE = False

process = psutil.Process(os.getpid())

In [12]:
def cpu_memory():
    return process.memory_info().rss / (1024**2)


def gpu_memory():
    if not GPU_AVAILABLE:
        return 0.0
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    return info.used / (1024**2)

In [4]:
from google.colab import files

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

Saving 42479-431756043.mp4 to 42479-431756043.mp4


In [10]:
def horn_schunck(prev, curr, alpha=1.0, iterations=60):

    prev = prev.astype(np.float32)
    curr = curr.astype(np.float32)

    Ix = cv2.Sobel(prev, cv2.CV_32F, 1, 0, 3)
    Iy = cv2.Sobel(prev, cv2.CV_32F, 0, 1, 3)
    It = curr - prev

    u = np.zeros_like(prev)
    v = np.zeros_like(prev)

    kernel = np.array([[0,1,0],
                       [1,0,1],
                       [0,1,0]], np.float32) / 4

    for _ in range(iterations):

        u_avg = cv2.filter2D(u,-1,kernel)
        v_avg = cv2.filter2D(v,-1,kernel)

        num = Ix*u_avg + Iy*v_avg + It
        den = alpha**2 + Ix**2 + Iy**2

        u = u_avg - Ix*num/den
        v = v_avg - Iy*num/den

    return u,v

In [13]:
def full_optical_pipeline(video_path):

    cap = cv2.VideoCapture(video_path)

    ret, first_frame = cap.read()
    first_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)

    # ===== Lucas-Kanade initial features (FIXED SETTINGS)
    lk_points = cv2.goodFeaturesToTrack(
        first_gray,
        maxCorners=3000,
        qualityLevel=0.001,
        minDistance=3,
        blockSize=7
    )

    hsv = np.zeros_like(first_frame)
    hsv[...,1] = 255

    frame_count = 0

    start = time.time()
    cpu_before = cpu_memory()
    gpu_before = gpu_memory()

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # ===================================================
        # 1️⃣ FARNEBACK DENSE FLOW
        # ===================================================
        flow = cv2.calcOpticalFlowFarneback(
            first_gray, gray, None,
            0.5,3,15,3,5,1.2,0
        )

        u_fb = flow[...,0]
        v_fb = flow[...,1]

        mag, ang = cv2.cartToPolar(u_fb,v_fb)

        # Motion segmentation
        motion_mask = mag > 2.0

        hsv[...,0] = ang*180/np.pi/2
        hsv[...,2] = cv2.normalize(mag,None,0,255,
                                   cv2.NORM_MINMAX)

        dense_rgb = cv2.cvtColor(hsv,cv2.COLOR_HSV2RGB)

        # ===================================================
        # 2️⃣ LUCAS-KANADE TRACKING
        # ===================================================
        if frame_count % 15 == 0:
            lk_points = cv2.goodFeaturesToTrack(
                gray,3000,0.001,3)

        lk_next, status, _ = cv2.calcOpticalFlowPyrLK(
            first_gray,gray,lk_points,None)

        tracked = frame.copy()

        good_new = lk_next[status==1]
        good_old = lk_points[status==1]

        for new,old in zip(good_new,good_old):

            a,b = new.ravel()
            c,d = old.ravel()

            cv2.line(tracked,(int(a),int(b)),
                     (int(c),int(d)),
                     (0,255,0),2)

            cv2.circle(tracked,(int(a),int(b)),
                       2,(0,0,255),-1)

        lk_points = good_new.reshape(-1,1,2)

        # ===================================================
        # 3️⃣ HORN-SCHUNCK (FIXED VISUALIZATION)
        # ===================================================
        u_hs,v_hs = horn_schunck(first_gray,gray)

        hs_mag = np.sqrt(u_hs**2 + v_hs**2)

        hs_vis = cv2.normalize(
            hs_mag,None,0,255,cv2.NORM_MINMAX)

        # ===================================================
        # SHOW OUTPUT
        # ===================================================
        if frame_count % 25 == 0:

            plt.figure(figsize=(16,4))

            plt.subplot(1,4,1)
            plt.imshow(dense_rgb)
            plt.title("Farneback Dense Flow")
            plt.axis("off")

            plt.subplot(1,4,2)
            plt.imshow(cv2.cvtColor(tracked,
                                    cv2.COLOR_BGR2RGB))
            plt.title("Lucas-Kanade Tracking")
            plt.axis("off")

            plt.subplot(1,4,3)
            plt.imshow(hs_vis,cmap='gray')
            plt.title("Horn-Schunck Flow")
            plt.axis("off")

            plt.subplot(1,4,4)
            plt.imshow(motion_mask,cmap='gray')
            plt.title("Motion Segmentation")
            plt.axis("off")

            plt.show()

        first_gray = gray.copy()

    cap.release()

    cpu_after = cpu_memory()
    gpu_after = gpu_memory()

    print("\n===== PERFORMANCE =====")
    print("Frames:",frame_count)
    print("Time taken:",round(time.time()-start,2),"sec")
    print("CPU memory used:",
          round(cpu_after-cpu_before,2),"MB")
    print("GPU memory used:",
          round(gpu_after-gpu_before,2),"MB")

In [14]:
full_optical_pipeline(video_path)

Output hidden; open in https://colab.research.google.com to view.